In [14]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/CJ-Mayes/SportsVizSunday/main/Data/Cricket/SVS%20May%202019%20-%20ICC%20Cricket%20World%20Cup.xls.xlsx"

df = pd.read_excel(url, sheet_name="Recent Team History")

# Additional columns
df["Win %"] = df["WonDescending"] / df["Mat"] * 100
df["Loss %"] = df["Lost"] / df["Mat"] * 100
df["Win-Loss Gap"] = df["Win %"] - df["Loss %"]

In [15]:
# Score range
chart1 = alt.Chart(df).mark_rule(size=8).encode(
    x=alt.X("LS:Q", title="Lowest score"),
    x2="HS:Q",
    y=alt.Y("Team:N", title=None),
    color=alt.Color("Win %:Q", title="Win %"),
    tooltip=["Team", "HS", "LS", "WonDescending", "Lost"]
).properties(
    title=alt.TitleParams(
        text="1. Score Range",
        subtitle="The range between each team's lowest and highest score",
        anchor="start"
    ),
    width=700,
    height=350
)

# Winning vs losing
chart2 = alt.Chart(
    df
).transform_fold(
    ["Win %", "Loss %"],
    as_=["Result", "Percentage"]
).mark_bar().encode(
    x=alt.X(
        "Percentage:Q",
        stack="normalize",
        title="Share of matches"
    ),
    y=alt.Y("Team:N", title=None),
    color=alt.Color("Result:N", title=None),
    tooltip=["Team", "Mat", "WonDescending", "Lost"]
).properties(
    title=alt.TitleParams(
        text="2. Winning vs Losing",
        subtitle="How each team's matches were divided between wins and losses",
        anchor="start"
    ),
    width=700,
    height=350
)

# W/L ratio vs scoring rate
chart3 = alt.Chart(df).mark_circle(size=120).encode(
    x=alt.X("W/L:Q", title="Win / Loss ratio"),
    y=alt.Y("RPO:Q", title="Runs per over"),
    size=alt.Size("Mat:Q", title="Matches"),
    color=alt.Color("Team:N", legend=None),
    tooltip=[
        "Team", "W/L", "RPO", "Mat",
        "WonDescending", "Lost"
    ]
).properties(
    title=alt.TitleParams(
        text="3. Winning Efficiency vs Scoring Rate",
        subtitle="Do the most successful teams also score quickly?",
        anchor="start"
    ),
    width=700,
    height=350
)

# W/L ratio vs batting average
chart4 = alt.Chart(df).mark_circle(size=120).encode(
    x=alt.X("W/L:Q", title="Win / Loss ratio"),
    y=alt.Y("Ave:Q", title="Batting average"),
    size=alt.Size("Mat:Q", title="Matches"),
    color=alt.Color("Team:N", legend=None),
    tooltip=[
        "Team", "W/L", "Ave", "RPO",
        "Mat", "WonDescending", "Lost"
    ]
).properties(
    title=alt.TitleParams(
        text="4. Winning Efficiency vs Batting Strength",
        subtitle="Relationship between winning record and batting average",
        anchor="start"
    ),
    width=700,
    height=350
)

# Team fingerprints
metrics = ["W/L", "Ave", "RPO", "HS"]

for col in metrics:
    df[col + "_norm"] = (
        (df[col] - df[col].min()) /
        (df[col].max() - df[col].min()) * 100
    )

fingerprint = df.melt(
    id_vars="Team",
    value_vars=[m + "_norm" for m in metrics],
    var_name="Metric",
    value_name="Score"
)

fingerprint["Metric"] = fingerprint["Metric"].str.replace(
    "_norm", "", regex=False
)

chart5 = alt.Chart(fingerprint).mark_line(
    point=True
).encode(
    x=alt.X("Metric:N", title=None),
    y=alt.Y(
        "Score:Q",
        title="Relative strength",
        scale=alt.Scale(domain=[0, 100])
    ),
    color=alt.Color(
        "Team:N",
        legend=alt.Legend(columns=2)
    ),
    tooltip=["Team", "Metric", "Score"]
).properties(
    title=alt.TitleParams(
        text="5. Team Fingerprints",
        subtitle="Each team's relative profile across four key metrics",
        anchor="start"
    ),
    width=700,
    height=350
)

# Cricket landscape
points = alt.Chart(df).mark_circle(size=120).encode(
    x=alt.X("W/L:Q", title="Win / Loss ratio"),
    y=alt.Y("RPO:Q", title="Runs per over"),
    color=alt.Color("Team:N", legend=None),
    tooltip=[
        "Team", "W/L", "RPO",
        "WonDescending", "Lost"
    ]
)

labels = alt.Chart(df).mark_text(
    align="left",
    dx=8,
    dy=-8
).encode(
    x="W/L:Q",
    y="RPO:Q",
    text="Team:N"
)

vertical = alt.Chart(
    pd.DataFrame({"x": [df["W/L"].median()]})
).mark_rule(
    strokeDash=[5, 5]
).encode(
    x="x:Q"
)

horizontal = alt.Chart(
    pd.DataFrame({"y": [df["RPO"].median()]})
).mark_rule(
    strokeDash=[5, 5]
).encode(
    y="y:Q"
)

chart6 = (
    points + labels + vertical + horizontal
).properties(
    title=alt.TitleParams(
        text="6. The Cricket Landscape",
        subtitle="Teams positioned by winning efficiency and scoring rate",
        anchor="start"
    ),
    width=700,
    height=350
)

# Combine all six
final_chart = alt.vconcat(
    chart1,
    chart2,
    chart3,
    chart4,
    chart5,
    chart6
).properties(
    title=alt.TitleParams(
        text="ICC Cricket World Cup — Team Performance",
        subtitle="Recent team history, 2015–2019",
        anchor="middle",
        fontSize=24,
        subtitleFontSize=16
    )
)

final_chart


alt.VConcatChart(...)